# Trabalho Prático: Tópicos Avançados em Inteligência Artificial
## Projeto: CardioRisk AI — Predição de Risco de Doença Cardiovascular
**Instituição:** Instituto Federal de São Paulo (IFSP) — Câmpus Catanduva  
**Docente:** Prof. Dr. Márcio Andrey Teixeira  
**Discente:** Pedro Henrique Oliveira Pereira  
**Data de Entrega:** 26/10/2026

---
### 1. Contexto e Definição do Problema
As doenças cardiovasculares constituem a principal causa de mortalidade no mundo. A identificação precoce de indivíduos sob risco permite intervenções profiláticas decisivas.
Este projeto desenvolve uma solução supervisionada de classificação binária baseada no dataset clínico de Cleveland (UCI Machine Learning Repository) para prever a presença de cardiopatia (estenose arterial > 50%), comparando **Regressão Logística** e **Random Forest** com avaliação rigorosa sem vazamento de dados (data leakage).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score, f1_score

print("Bibliotecas importadas com sucesso!")

### 2. Carga e Análise Exploratória dos Dados (EDA)
O dataset possui 303 pacientes e 14 variáveis. A variável alvo original possui valores de 0 (ausência) a 4 (grau de estreitamento). Convertemos em classificação binária (0: ausência, 1: presença).

In [2]:
df = pd.read_csv("../data/heart.csv")
df["target"] = (df["target"] > 0).astype(int)
display(df.head())
print("Informações do Dataset:")
df.info()

### 3. Divisão Treino e Teste sem Data Leakage
Conforme orientação da Seção 2 do edital, separamos o conjunto de teste (20%) antes de qualquer transformação estatística.

In [3]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Tamanho de Treino: {len(X_train)} registros")
print(f"Tamanho de Teste (Cego): {len(X_test)} registros")

### 4. Construção do Pipeline de Pré-processamento e Modelos
- Imputação de nulos por mediana para numéricas e moda para categóricas.
- Padronização (StandardScaler) para variáveis numéricas.
- Codificação One-Hot para categóricas.

In [4]:
num_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
cat_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

num_trans = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_trans = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False))])

preproc = ColumnTransformer([("num", num_trans, num_features), ("cat", cat_trans, cat_features)])

pipe_lr = Pipeline([("preproc", preproc), ("clf", LogisticRegression(C=1.0, max_iter=1000, random_state=42))])
pipe_rf = Pipeline([("preproc", preproc), ("clf", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))])

### 5. Validação Cruzada 5-Fold Estratificada (Treinamento)

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_lr = cross_validate(pipe_lr, X_train, y_train, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "roc_auc"])
cv_rf = cross_validate(pipe_rf, X_train, y_train, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "roc_auc"])

print(f"LR CV Acurácia: {cv_lr['test_accuracy'].mean():.4f} | F1: {cv_lr['test_f1'].mean():.4f} | ROC-AUC: {cv_lr['test_roc_auc'].mean():.4f}")
print(f"RF CV Acurácia: {cv_rf['test_accuracy'].mean():.4f} | F1: {cv_rf['test_f1'].mean():.4f} | ROC-AUC: {cv_rf['test_roc_auc'].mean():.4f}")

### 6. Avaliação no Conjunto de Teste Cego

In [6]:
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

y_pred_lr = pipe_lr.predict(X_test)
y_pred_rf = pipe_rf.predict(X_test)

print("=== Relatório Regressão Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=["Saudavel", "Cardiopatia"]))
print("=== Relatório Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=["Saudavel", "Cardiopatia"]))

### 7. Exportação do Pipeline Campeão para Produção
A Regressão Logística é selecionada pelo F1-Score consistente (0.8621), ROC-AUC elevado (0.9621) e alta interpretabilidade clínica.

In [7]:
joblib.dump(pipe_lr, "../models/heart_disease_pipeline.joblib")
print("Pipeline serializado com sucesso em ../models/heart_disease_pipeline.joblib!")